# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyamT/flyrank_new/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. What one row means.** In the warehouse, one row is one page (`content_hash_id`) of one client (`client_hash_id`) on one day (`report_date`), verified with 0 duplicate keys. For my model, I roll the days up to one row per page.

In [6]:
!pip install -q datasets duckdb
from datasets import load_dataset
from google.colab import userdata

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance",
                  streaming=True, split="train", token=userdata.get("HF_TOKEN"))
row = next(iter(ds))
for name, value in row.items():
    print(f"{name}: {type(value).__name__}")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

report_date: date
client_hash_id: str
content_hash_id: str
client_has_gsc: bool
client_has_ga4: bool
gsc_data_available: bool
ga4_data_available: bool
gsc_impressions: int
gsc_clicks: int
gsc_sum_position: int
gsc_avg_position: float
ga4_pageviews: int
ga4_sessions: int
ga4_users: int
ga4_engaged_sessions: int
ga4_total_engagement_sec: int
sessions_organic: int
sessions_direct: int
sessions_referral: int
sessions_social: int
sessions_paid: int
sessions_ai: int
ai_chatgpt: int
ai_perplexity: int
ai_gemini: int
ai_copilot: int
ai_claude: int
ai_meta: int
ai_other: int
scroll_events: int


In [2]:
!pip install -q huggingface_hub duckdb
import duckdb
from huggingface_hub import list_repo_files, hf_hub_download
from google.colab import userdata   # VS Code: import os; token = os.environ["HF_TOKEN"]

token = userdata.get("HF_TOKEN")
REPO = "FlyRank/internship-warehouse"

files = list_repo_files(REPO, repo_type="dataset", token=token)
march = [f for f in files if "fact_content_daily_performance" in f
         and "month=2026-03" in f and f.endswith(".parquet")]
print("March files:", march)

local = [hf_hub_download(REPO, f, repo_type="dataset", token=token) for f in march]
con = duckdb.connect()
con.sql(f"CREATE VIEW t AS SELECT * FROM read_parquet({local})")
print(con.sql("SELECT COUNT(*) AS rows FROM t").df())

March files: ['fact_content_daily_performance/month=2026-03/data_0.parquet']


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

      rows
0  9841378


In [3]:
# Query 1: grain. duplicate_keys should be 0 if one row = one page, one client, one day
print(con.sql("""
SELECT COUNT(*) AS rows_total,
       (SELECT COUNT(*) FROM (SELECT 1 FROM t
          GROUP BY report_date, client_hash_id, content_hash_id
          HAVING COUNT(*) > 1)) AS duplicate_keys
FROM t""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_total  duplicate_keys
0     9841378               0


In [4]:
# Query 2: size and date span of my slice
print(con.sql("""
SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS pages
FROM t""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows  first_day   last_day  clients   pages
0  9841378 2026-03-01 2026-03-31       55  331437


In [5]:
# Query 3: availability, using IS TRUE (flags can be NULL)
print(con.sql("""
SELECT COUNT(*) AS rows_total,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)     AS gsc_true,
       COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL)     AS gsc_null
FROM t""").df())

   rows_total  gsc_true  gsc_not_true  gsc_null
0     9841378   3611061       6230317         0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.